### Cel projektu:
Notebook zawiera analizę statystyczną danych z eksperymentu kontrolowanego (RCT), mającego na celu ocenę skuteczności interwencji cyfrowej w nauczaniu matematyki. Badanie objęło 1224 uczniów ze 161 szkół, a pomiary wykonano w dwóch etapach: przed (pretest) i po interwencji (posttest).

### Hipoteza badawcza:
Uczniowie z grupy eksperymentalnej osiągają istotnie wyższe wyniki w postteście niż uczniowie z grupy kontrolnej, przy uwzględnieniu ich poziomu startowego (pretestu).

### Spis treści:
1. Weryfikacja danych
2. Eksploracja danych
3. Analiza czynnikowa
4. CTT - Classical Test Theory
5. IRT - Item Response Theory
6. Wskaźnik umiejętności (theta)
7. Weryfikacja hipotezy

In [97]:
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo
from factor_analyzer import FactorAnalyzer
import pingouin as pg
from girth import twopl_mml, threepl_mml, ability_eap
import statsmodels.formula.api as smf

import warnings
warnings.filterwarnings('ignore')

In [98]:
DATA_PATH = Path("task/math_data.csv")
data = pd.read_csv(DATA_PATH, sep=",")

RANDOM_SEED = 4321
np.random.seed(RANDOM_SEED)

In [99]:
PreTestData = data[data["pomiar"] == 1]
PostTestData = data[data["pomiar"] == 2]

NonMatCols = [c for c in data.columns if not c.startswith("mat_")]
MatCols = [c for c in data.columns if c.startswith("mat_")]

def _mat_sort_key(name):
    """Sort key: mat_1, mat_2, ..., mat_9, mat_10 (numeric order)."""
    n = name.replace("mat_", "")
    return int(n) if n.isdigit() else 0

### Weryfikacja danych

Weryfikacja danych potwierdza, że:
- w kolumnach kluczowych dla analizy nie znaleziono żadnych braków danych. Zbiór jest gotowy do modelowania bez dodatkowych imputacji.
- randomizacja na poziomie szkół przebiegła poprawnie - żadna szkoła nie została przypisana do obu grup jednocześnie.
- wszyscy uczestnicy etapu PreTest wzięli również udział w etapie PostTest.
- NULL w odpowiedzi na dane zadanie oznacza, że zadanie nie było wykorzystane w danym pomiarze, a nie że dane są nie pełne.

In [100]:
nulls_non_mat = data[NonMatCols].isna().any(axis=1).sum()
print(f"Liczba brakujących wartości w kolumnach innych niż mat_*: {nulls_non_mat}")

Liczba brakujących wartości w kolumnach innych niż mat_*: 0


In [101]:
TasksInPreTest = PreTestData[MatCols].notna().any(axis=0)
TasksInPostTest = PostTestData[MatCols].notna().any(axis=0)
print("Zadania z danymi w PreTest:", TasksInPreTest[TasksInPreTest].index.tolist())
print("Zadania z danymi w PostTest:", TasksInPostTest[TasksInPostTest].index.tolist())

Zadania z danymi w PreTest: ['mat_9', 'mat_10', 'mat_11', 'mat_12', 'mat_13', 'mat_14', 'mat_15', 'mat_16', 'mat_17', 'mat_18', 'mat_19', 'mat_20', 'mat_21', 'mat_22', 'mat_23', 'mat_33', 'mat_34', 'mat_35', 'mat_36', 'mat_37', 'mat_38', 'mat_39', 'mat_40', 'mat_32']
Zadania z danymi w PostTest: ['mat_1', 'mat_2', 'mat_3', 'mat_4', 'mat_5', 'mat_6', 'mat_7', 'mat_8', 'mat_24', 'mat_25', 'mat_26', 'mat_27', 'mat_28', 'mat_29', 'mat_30', 'mat_31', 'mat_41', 'mat_42', 'mat_43', 'mat_44', 'mat_45', 'mat_46', 'mat_47', 'mat_48']


In [102]:
NGroupsPerSchool = data.groupby("id_szkoly")["grupa"].nunique()

print("Weryfikacja randomizacji")
print(f"Liczba szkół: {data['id_szkoly'].nunique()}")
print(f"Szkół z więcej niż jedną grupą: {len(NGroupsPerSchool[NGroupsPerSchool > 1])}")

Weryfikacja randomizacji
Liczba szkół: 161
Szkół z więcej niż jedną grupą: 0


In [103]:
PreTestIDs = set(data[data["pomiar"] == 1]["id_ucznia"])
PostTestIDs = set(data[data["pomiar"] == 2]["id_ucznia"])

print("Uczniowie: PreTest vs PostTest")
print(f"Uczniów tylko w PreTest (brak w PostTest): {len(PreTestIDs - PostTestIDs)}")
print(f"Uczniów tylko w PostTest (brak w PreTest): {len(PostTestIDs - PreTestIDs)}")

Uczniowie: PreTest vs PostTest
Uczniów tylko w PreTest (brak w PostTest): 0
Uczniów tylko w PostTest (brak w PreTest): 0


### Eksploracja danych

Eksploracja danych pokazuje, że:
- liczebność grup jest dość zrównoważona i niewielka różnica w liczebności (ok. 7%) nie powinna wpływać negatywnie na stabilność estymacji.
- żadne z zadań nie okazało się za łatwe, lub za trudne, co oznacza, że każde pytanie wnosi unikalną informację do modelu o różnicowaniu uczniów
- średni wynik dla zadań wynosi ok. 0.5, czyli testy zostały dobrze skalibrowane pod kątem populacji uczniów.

In [104]:
NSchools = data["id_szkoly"].nunique()
NStudents = data["id_ucznia"].nunique()

StudentSchoolGroupMatrix = (
    data.groupby("grupa")
    .agg(NSchools=("id_szkoly", "nunique"), NStudents=("id_ucznia", "nunique"))
    .rename(index={0: "kontrolna", 1: "eksperymentalna"})
)
StudentSchoolGroupMatrix.index.name = "grupa"
StudentSchoolGroupMatrix.loc["TOTAL"] = [NSchools, NStudents]

print("Macierz: szkoły i uczniowie w podziale na grupę")
print(StudentSchoolGroupMatrix)

Macierz: szkoły i uczniowie w podziale na grupę
                 NSchools  NStudents
grupa                               
kontrolna              77        591
eksperymentalna        84        633
TOTAL                 161       1224


In [105]:
PreTestTasksList = TasksInPreTest[TasksInPreTest].index.tolist()
PostTestTasksList = TasksInPostTest[TasksInPostTest].index.tolist()

PreTestData_num = PreTestData[PreTestTasksList].apply(pd.to_numeric, errors="coerce")
PostTestData_num = PostTestData[PostTestTasksList].apply(pd.to_numeric, errors="coerce")

PreTestDifficulty = PreTestData_num.mean().reindex(
    sorted(PreTestData_num.columns, key=_mat_sort_key)
)
PostTestDifficulty = PostTestData_num.mean().reindex(
    sorted(PostTestData_num.columns, key=_mat_sort_key)
)

In [106]:
LOWER_MAT_THRESHOLD, UPPER_MAT_THRESHOLD = 0.05, 0.95

def plot_trudnosc_bars(trudnosc_series, title):
    fig = px.bar(x=trudnosc_series.index, y=trudnosc_series.values, title=title, labels={"x": "Zadanie", "y": "Średni wynik"})
    fig.update_yaxes(range=[0, 1])
    fig.add_hline(y=UPPER_MAT_THRESHOLD, line_dash="dash", line_color="red")
    fig.add_hline(y=LOWER_MAT_THRESHOLD, line_dash="dash", line_color="red")
    fig.show()


plot_trudnosc_bars(PreTestDifficulty, "Trudność zadań – PreTest (średni wynik)")
plot_trudnosc_bars(PostTestDifficulty, "Trudność zadań – PostTest (średni wynik)")

In [107]:
print("Statystiki opisowe zadań PreTest")
print(PreTestData_num.describe().T)

Statystiki opisowe zadań PreTest
         count      mean       std  min  25%  50%  75%  max
mat_9   1224.0  0.584150  0.493069  0.0  0.0  1.0  1.0  1.0
mat_10  1224.0  0.597222  0.490657  0.0  0.0  1.0  1.0  1.0
mat_11  1224.0  0.415033  0.492929  0.0  0.0  0.0  1.0  1.0
mat_12  1224.0  0.560458  0.496534  0.0  0.0  1.0  1.0  1.0
mat_13  1224.0  0.426471  0.494766  0.0  0.0  0.0  1.0  1.0
mat_14  1224.0  0.513072  0.500033  0.0  0.0  1.0  1.0  1.0
mat_15  1224.0  0.617647  0.486161  0.0  0.0  1.0  1.0  1.0
mat_16  1224.0  0.674020  0.468932  0.0  0.0  1.0  1.0  1.0
mat_17  1224.0  0.529412  0.499338  0.0  0.0  1.0  1.0  1.0
mat_18  1224.0  0.569444  0.495356  0.0  0.0  1.0  1.0  1.0
mat_19  1224.0  0.656046  0.475220  0.0  0.0  1.0  1.0  1.0
mat_20  1224.0  0.402778  0.490657  0.0  0.0  0.0  1.0  1.0
mat_21  1224.0  0.754085  0.430804  0.0  1.0  1.0  1.0  1.0
mat_22  1224.0  0.635621  0.481452  0.0  0.0  1.0  1.0  1.0
mat_23  1224.0  0.564542  0.496019  0.0  0.0  1.0  1.0  1.0
mat_33 

In [108]:
print("Statystiki opisowe zadań PostTest")
print(PostTestData_num.describe().T)

Statystiki opisowe zadań PostTest
         count      mean       std  min  25%  50%  75%  max
mat_1   1224.0  0.399510  0.489998  0.0  0.0  0.0  1.0  1.0
mat_2   1224.0  0.787582  0.409186  0.0  1.0  1.0  1.0  1.0
mat_3   1224.0  0.653595  0.476019  0.0  0.0  1.0  1.0  1.0
mat_4   1224.0  0.596405  0.490819  0.0  0.0  1.0  1.0  1.0
mat_5   1224.0  0.376634  0.484740  0.0  0.0  0.0  1.0  1.0
mat_6   1224.0  0.705882  0.455831  0.0  0.0  1.0  1.0  1.0
mat_7   1224.0  0.615196  0.486748  0.0  0.0  1.0  1.0  1.0
mat_8   1224.0  0.522876  0.499681  0.0  0.0  1.0  1.0  1.0
mat_24  1224.0  0.777778  0.415910  0.0  1.0  1.0  1.0  1.0
mat_25  1224.0  0.725490  0.446449  0.0  0.0  1.0  1.0  1.0
mat_26  1224.0  0.624183  0.484531  0.0  0.0  1.0  1.0  1.0
mat_27  1224.0  0.359477  0.480043  0.0  0.0  0.0  1.0  1.0
mat_28  1224.0  0.442810  0.496922  0.0  0.0  0.0  1.0  1.0
mat_29  1224.0  0.264706  0.441357  0.0  0.0  0.0  1.0  1.0
mat_30  1224.0  0.718954  0.449694  0.0  0.0  1.0  1.0  1.0
mat_31

### Analiza czynnikowa

Analiza czynnikowa pokazuje, że:
- struktura korelacji między zadaniami jest wystarczająco silna, aby wyodrębnić z nich wspólny czynnik. Wyniki Bartletta potwierdza istotną korelacje, a wskaźnik KMO potwierdza silną korelacje między zadaniami.
- testy są jednowymiarowe. W obu pomiarach widać gwałtowny spadek wartości po pierwszym czynniku, co pozwala na bezpieczne stosowanie modeli IRT.

In [109]:
_, p_value = calculate_bartlett_sphericity(PreTestData_num)
_, kmo_model = calculate_kmo(PreTestData_num)
print(f"PreTest - Bartlett p-value: {p_value:.4f}, KMO: {kmo_model:.4f}")

_, p_value = calculate_bartlett_sphericity(PostTestData_num)
_, kmo_model = calculate_kmo(PostTestData_num)
print(f"PostTest - Bartlett p-value: {p_value:.4f}, KMO: {kmo_model:.4f}")

PreTest - Bartlett p-value: 0.0000, KMO: 0.9193
PostTest - Bartlett p-value: 0.0000, KMO: 0.9465


In [110]:
def create_scree_plot(data, title):
    fa = FactorAnalyzer(rotation=None)
    fa.fit(data)
    ev, _ = fa.get_eigenvalues()
    
    factors = list(range(1, len(ev) + 1))
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=factors,
        y=ev,
        mode="lines+markers",
        name="Wartości własne",
        marker=dict(size=10, color="royalblue"),
        line=dict(width=3),
    ))
    fig.add_shape(
        type="line",
        x0=1, y0=1, x1=max(factors), y1=1,
        line=dict(color="red", width=2, dash="dash"),
    )
    fig.add_annotation(
        x=max(factors) * 0.9, y=1.2,
        text="Kryterium Kaisera (EV=1)",
        showarrow=False,
        font=dict(color="red"),
    )
    fig.update_layout(
        title=title,
        xaxis_title="Numer czynnika",
        yaxis_title="Wartość własna (Eigenvalue)",
        template="plotly_white",
        xaxis=dict(tickmode="linear", tick0=1, dtick=1),
        hovermode="x unified",
    )

    fig.show()

In [111]:
create_scree_plot(PreTestData_num, title="Scree Plot - PreTest")

In [112]:
create_scree_plot(PostTestData_num, title="Scree Plot - PostTest")

### CTT - Classical Test Theory

Analiza rzetelności i mocy dyskryminacyjnej pokazuje, że:
- test jest rzetelny, co potwierdzają wartości Alfa Cronbacah powyżej 0.8. Wzrost wskaźnika w PostTest sugeruje jeszcze lepsze dopasowanie zadań do zróżnicowania poziomu umiejętności uczniów po zakończeniu programu.
- zadania bardzo dobrze odróżniają uczniów o wysokich umiejętnościach od uczniów słabszych. Wysoka moc dyskryminacyjna wielu zadań (powyżej 0.5) potwierdza, że odpowiedzi na poszczególne pytania są spójne z ogólnym wynikiem testu. 

In [113]:
PreTestAlpha = pg.cronbach_alpha(data=PreTestData_num)
print(f"Alfa Cronbacha (PreTest): {PreTestAlpha[0]:.3f} (CI: {PreTestAlpha[1]})")

PostTestAlpha = pg.cronbach_alpha(data=PostTestData_num)
print(f"Alfa Cronbacha (PostTest): {PostTestAlpha[0]:.3f}")

Alfa Cronbacha (PreTest): 0.821 (CI: [0.807 0.836])
Alfa Cronbacha (PostTest): 0.868


In [114]:
PreTestTotalScores = PreTestData_num.sum(axis=1)
PreTestItemDiscrimination = PreTestData_num.apply(lambda col: col.corr(PreTestTotalScores))

print("Moc dyskryminacyjna (Top 5 zadań - PreTest):")
print(PreTestItemDiscrimination.sort_values(ascending=False).head())

Moc dyskryminacyjna (Top 5 zadań - PreTest):
mat_34    0.579551
mat_35    0.521227
mat_10    0.509545
mat_37    0.508899
mat_17    0.501511
dtype: float64


### IRT - Item Response Theory

Przeprowadzenie modelowania IRT pokazuje, że:
- w modelu 3PL oszacowane prawdopodobieństwo poprawnej odpowiedzi przez zgadywanie dla wszystkich zadań jest bliskie 0. To skłania do wykorzystywania modelu 2PL w dalszych analizach
- test jest świetnie zbalansowany, czyli składa się z zadań łatwiejszych (przesuniętych w lewo na wykresie ICC) i zadań trudniejszych (przesuniętych w prawo).
- w teście występują zadania, które dobrze różnicują uczniów z różnymi poziomami umiejętności.
- test dostarcza najwięcej informacji o przeciętnym uczniu, bo większość krzywych przecina się w okoliach theta = 0

In [115]:
PreTestMatrixTransp = PreTestData_num.to_numpy(dtype=int).T
PostTestMatrixTransp = PostTestData_num.to_numpy(dtype=int).T

PreTest2PLEstimates = twopl_mml(PreTestMatrixTransp)
PostTest2PLEstimates = twopl_mml(PostTestMatrixTransp)

PreTest2PLItems = pd.DataFrame({
    'Trudność (b)': PreTest2PLEstimates['Difficulty'],
    'Dyskryminacja (a)': PreTest2PLEstimates['Discrimination'],
}, index=PreTestData_num.columns).reindex(sorted(PreTestData_num.columns, key=_mat_sort_key))

PostTest2PLItems = pd.DataFrame({
    'Trudność (b)': PostTest2PLEstimates['Difficulty'],
    'Dyskryminacja (a)': PostTest2PLEstimates['Discrimination'],
}, index=PostTestData_num.columns).reindex(sorted(PostTestData_num.columns, key=_mat_sort_key))

In [116]:
print("Parametry IRT - 2PL (PreTest):")
print(PostTest2PLItems)

Parametry IRT - 2PL (PreTest):
        Trudność (b)  Dyskryminacja (a)
mat_1       0.341121           1.921195
mat_2      -1.120449           1.747878
mat_3      -0.707307           1.123049
mat_4      -0.416759           1.206199
mat_5       0.543160           1.184794
mat_6      -0.740576           1.838372
mat_7      -0.477982           1.300522
mat_8      -0.087380           1.467214
mat_24     -0.953126           2.344874
mat_25     -1.103589           1.082669
mat_26     -0.612387           1.000004
mat_27      0.686499           1.021384
mat_28      0.345155           0.748880
mat_29      0.804603           2.171921
mat_30     -0.948767           1.302582
mat_31     -1.592765           1.977434
mat_41      0.279841           0.855788
mat_42     -0.321184           1.220446
mat_43     -0.013568           1.267921
mat_44      0.880659           1.417735
mat_45      1.187701           0.882863
mat_46     -0.246387           0.845564
mat_47     -0.242117           1.649245
mat_48   

In [117]:
print("\nParametry IRT - 2PL (PostTest):")
print(PostTest2PLItems)


Parametry IRT - 2PL (PostTest):
        Trudność (b)  Dyskryminacja (a)
mat_1       0.341121           1.921195
mat_2      -1.120449           1.747878
mat_3      -0.707307           1.123049
mat_4      -0.416759           1.206199
mat_5       0.543160           1.184794
mat_6      -0.740576           1.838372
mat_7      -0.477982           1.300522
mat_8      -0.087380           1.467214
mat_24     -0.953126           2.344874
mat_25     -1.103589           1.082669
mat_26     -0.612387           1.000004
mat_27      0.686499           1.021384
mat_28      0.345155           0.748880
mat_29      0.804603           2.171921
mat_30     -0.948767           1.302582
mat_31     -1.592765           1.977434
mat_41      0.279841           0.855788
mat_42     -0.321184           1.220446
mat_43     -0.013568           1.267921
mat_44      0.880659           1.417735
mat_45      1.187701           0.882863
mat_46     -0.246387           0.845564
mat_47     -0.242117           1.649245
mat_48 

In [118]:
def create_icc_curve(theta, a, b, c=None):
    """Uniwersalna krzywa ICC: 2PL (c=None) lub 3PL (c=Guessing)."""
    p = 1 / (1 + np.exp(-a * (theta - b)))
    if c is not None:
        p = c + (1 - c) * p
    return p


def plot_icc_curves(theta, curves, title, y_floor=0):
    fig = go.Figure()
    for curve in curves:
        c = curve.get("c")
        y = create_icc_curve(theta, curve["a"], curve["b"], c=c)
        fig.add_trace(
            go.Scatter(
                x=theta,
                y=y,
                mode="lines",
                name=curve["label"],
                line=dict(width=2),
            )
        )
    fig.add_vline(x=0, line_dash="dot", line_color="gray")
    fig.add_hline(y=0.5, line_dash="dot", line_color="gray")
    if y_floor > 0:
        fig.add_hline(y=y_floor, line_dash="dot", line_color="gray", annotation_text="c (Guessing)")
    fig.update_layout(
        title=title,
        xaxis_title="θ",
        yaxis_title="P(poprawna odpowiedzi)",
        yaxis=dict(range=[y_floor, 1]),
    )
    fig.show()

In [119]:
PreTest3PLEstimates = threepl_mml(PreTestMatrixTransp)
PostTest3PLEstimates = threepl_mml(PostTestMatrixTransp)

PreTest3PLItems = pd.DataFrame({
    'Trudność (b)': PreTest3PLEstimates['Difficulty'],
    'Dyskryminacja (a)': PreTest3PLEstimates['Discrimination'],
    'Prawdopodobieństwo trafienia (c)': PreTest3PLEstimates['Guessing'],
}, index=PreTestData_num.columns).reindex(sorted(PreTestData_num.columns, key=_mat_sort_key))

PostTest3PLItems = pd.DataFrame({
    'Trudność (b)': PostTest3PLEstimates['Difficulty'],
    'Dyskryminacja (a)': PostTest3PLEstimates['Discrimination'],
    'Prawdopodobieństwo trafienia (c)': PostTest3PLEstimates['Guessing'],
}, index=PostTestData_num.columns).reindex(sorted(PostTestData_num.columns, key=_mat_sort_key))

In [120]:
print("Parametry IRT - 3PL (PreTest):")
print(PreTest3PLItems)

Parametry IRT - 3PL (PreTest):
        Trudność (b)  Dyskryminacja (a)  Prawdopodobieństwo trafienia (c)
mat_9      -0.508936           0.751455                      1.490116e-08
mat_10     -0.324627           1.344874                      4.002443e-02
mat_11      0.894575           1.218586                      1.647303e-01
mat_12     -0.393242           0.683308                      1.475215e-08
mat_13      0.729630           0.872907                      9.488586e-02
mat_14      0.083620           1.149127                      6.172387e-02
mat_15     -0.345562           0.644862                      1.490165e-01
mat_16     -0.319085           1.668225                      2.079234e-01
mat_17     -0.127501           1.182397                      2.935678e-08
mat_18     -0.090031           0.956489                      1.067064e-01
mat_19     -0.348310           1.290448                      1.728364e-01
mat_20      1.147132           0.994925                      1.723994e-01
mat_21 

In [121]:
print("\nParametry IRT - 3PL (PostTest):")
print(PostTest3PLItems)


Parametry IRT - 3PL (PostTest):
        Trudność (b)  Dyskryminacja (a)  Prawdopodobieństwo trafienia (c)
mat_1       0.343980           1.967901                      3.009478e-03
mat_2      -1.058524           1.787829                      4.718071e-02
mat_3      -0.709499           1.117741                      2.965331e-08
mat_4      -0.168781           1.419635                      1.162060e-01
mat_5       0.540712           1.193279                      2.965421e-08
mat_6      -0.741152           1.834963                      5.799538e-08
mat_7      -0.479613           1.292999                      5.871652e-08
mat_8      -0.087537           1.462696                      7.285579e-08
mat_24     -0.950601           2.344501                      2.724942e-03
mat_25     -0.994429           1.109229                      5.784338e-02
mat_26     -0.089691           1.316554                      2.136148e-01
mat_27      0.887919           1.667135                      1.253518e-01
mat_2

#### Przykładowe wykresy

In [122]:
theta = np.linspace(-3, 3, 301)

plot_icc_curves(
    theta,
    [
        {"a": 1.5, "b": -1, "label": "Łatwe (b = -1)"},
        {"a": 1.5, "b": 0, "label": "Średnie (b = 0)"},
        {"a": 1.5, "b": 1.5, "label": "Trudne (b = 1.5)"},
    ],
    title="2PL: Położenie krzywej (Trudność b)",
)

plot_icc_curves(
    theta,
    [
        {"a": 0.5, "b": 0, "label": "Słaba (a = 0.5)"},
        {"a": 1.5, "b": 0, "label": "Średnia (a = 1.5)"},
        {"a": 2.5, "b": 0, "label": "Silna (a = 2.5)"},
    ],
    title="2PL: Nachylenie krzywej (Dyskryminacja a)",
)

plot_icc_curves(
    theta,
    [
        {"a": 1.5, "b": 0, "label": "2PL (c = 0)"},
        {"a": 1.5, "b": 0, "c": 0.2, "label": "3PL (c = 0.2)"},
        {"a": 1.5, "b": 0, "c": 0.25, "label": "3PL (c = 0.25)"},
    ],
    title="3PL: Punkt startowy (Prawdopodobieństwo trafienia c) – szansa na trafienie przy bardzo niskim θ",
    y_floor=0.25,
)

#### Wykresy z danych eksperymentu

In [123]:
PreTest2PLCurves = [
    {
        "a": row["Dyskryminacja (a)"],
        "b": row["Trudność (b)"],
        "label": idx,
    }
    for idx, row in PreTest2PLItems.iterrows()
]
plot_icc_curves(
    theta,
    PreTest2PLCurves,
    title="ICC dla wszystkich zadań PreTestu (2PL)",
)

In [130]:
PostTest2PLCurves = [
    {
        "a": row["Dyskryminacja (a)"],
        "b": row["Trudność (b)"],
        "label": idx,
    }
    for idx, row in PostTest2PLItems.iterrows()
]
plot_icc_curves(
    theta,
    PostTest2PLCurves,
    title="ICC dla wszystkich zadań PostTestu (2PL)",
)

In [124]:
PreTest3PLCurves = [
    {
        "a": row["Dyskryminacja (a)"],
        "b": row["Trudność (b)"],
        "c": row["Prawdopodobieństwo trafienia (c)"],
        "label": idx,
    }
    for idx, row in PreTest3PLItems.iterrows()
]
plot_icc_curves(
    theta,
    PreTest3PLCurves,
    title="ICC dla wszystkich zadań PreTestu (3PL)",
    y_floor=0,
)

### Wskaźnik umiejętności (theta)

Na podstawie parametrów modelu 2PL wyestymowano indywidualne wyniki uczniów (wskaźniki theta) metodą EAP (Expected A Posteriori). Wyniki zostały poddane standaryzacji (z-score), co pozwala na bezpośrednie porównanie obu pomiarów. Estymacja wyników pokazuje, że:
- standaryzacja przebiegła pomyślnie (średnia = 0, std. = 1), co jest kluczowe dal interpretacji siły efektu w modelu regresji
- rozkłady umiejętności są symetryczne, lekko spłaszczone. To sugeruje, że testy dobrze różnicowały uczniów w szerokim spektrum umiejętności, nie skupiając wszystkich wyników wyłącznie wokół średniej
- mimo wizualnych zniekształceń rozkładu PostTest, statystyki opisowe potwierdzają, że wskaźniki theta mają dobre właściwości psychometryczne i nadają się do końcowej werfikacyjni hipotezy badawczej.

In [125]:
PreTestThetaEstimates = ability_eap(
    dataset=PreTestMatrixTransp, 
    difficulty=PreTest2PLEstimates['Difficulty'], 
    discrimination=PreTest2PLEstimates['Discrimination']
)
PostTestThetaEstimates= ability_eap(
    dataset=PostTestMatrixTransp, 
    difficulty=PostTest2PLEstimates['Difficulty'], 
    discrimination=PostTest2PLEstimates['Discrimination']
)

PreTestThetaEstimatesScaled = (PreTestThetaEstimates - PreTestThetaEstimates.mean()) / PreTestThetaEstimates.std()
PostTestThetaEstimatesScaled = (PostTestThetaEstimates - PostTestThetaEstimates.mean()) / PostTestThetaEstimates.std()

PreTestSeriesWithTheta = pd.Series(PreTestThetaEstimatesScaled, index=PreTestData_num.index, name='theta_pretest_z')
PostTestSeriesWithTheta = pd.Series(PostTestThetaEstimatesScaled, index=PostTestData_num.index, name='theta_posttest_z')

In [126]:
PreTestInfo = (
    data.loc[PreTestData_num.index, ['id_szkoly', 'id_ucznia', 'grupa']]
    .assign(theta_pretest_z=PreTestSeriesWithTheta.values)
)

PostTestInfo = (
    data.loc[PostTestData_num.index, ['id_ucznia']]
    .assign(theta_posttest_z=PostTestSeriesWithTheta.values)
)

TableWithThetaEstimatesPerStudent = pd.merge(
    PreTestInfo[['id_ucznia', 'id_szkoly', 'grupa', 'theta_pretest_z']],
    PostTestInfo[['id_ucznia', 'theta_posttest_z']],
    on='id_ucznia',
    how='left'
)

In [127]:
def theta_describe(z, name):
    s = pd.Series(z)
    print(f"{name}:  średnia = {s.mean():.4f},  SD = {s.std():.4f},  skośność = {s.skew():.4f},  kurtoza = {s.kurtosis():.4f}")

theta_describe(PreTestThetaEstimatesScaled, "Theta PreTest (z)")
theta_describe(PostTestThetaEstimatesScaled, "Theta PostTest (z)")

def plot_theta_histogram(z, title):
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=z, nbinsx=25))
    fig.add_vline(x=0, line_dash="dash", line_color="red", annotation_text="średnia = 0")
    fig.update_layout(title=title, xaxis_title="θ (z)", yaxis_title="Liczba")
    fig.show()


plot_theta_histogram(PreTestThetaEstimatesScaled, "Rozkład theta PreTest (z-score)")
plot_theta_histogram(PostTestThetaEstimatesScaled, "Rozkład theta PostTest (z-score)")

Theta PreTest (z):  średnia = -0.0000,  SD = 1.0004,  skośność = 0.0233,  kurtoza = -0.4135
Theta PostTest (z):  średnia = -0.0000,  SD = 1.0004,  skośność = -0.1030,  kurtoza = -0.2997


### Weryfikacja hipotezy

hipoteza badawcza
```
Poziom umiejętności matematycznych uczniów w drugim pomiarze jest wyższy 
w grupie eksperymentalnej niż w grupie kontrolnej, przy kontroli poziomu 
umiejętności matematycznych przed interwencją.
```

Estymacja modelu mieszanego w celu zbadania hipotezy pokazuje, że:
- nie ma istotnego wpływu wykorzystania narzędzi cyfrowych w nauce matematyki. Efekt grupy jest bardzo niski i nie istotny statystycznie. Różnica między grupą kontrolną a eksperymentalną jest nie istotnie różna od zera.
- najistotniejszym czynnikiem jest poziom wiedzy przez przystąpieniem do eksperymentu. Wynik  końcowy ucznia jest w 70% zdeterminowany prze jego wiedzę przed rozpoczęciem programu.
- znaczna część zmienności wyników zależy od specyfiki konkretnej szkoły, co w pełni uzasadnia zastosowanie modelu mieszanego.

In [128]:
MODEL_FORMULA = "theta_posttest_z ~ grupa + theta_pretest_z"

model = smf.mixedlm(formula=MODEL_FORMULA, data=TableWithThetaEstimatesPerStudent, groups=TableWithThetaEstimatesPerStudent["id_szkoly"])
result = model.fit()

In [129]:
print(result.summary())

            Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: theta_posttest_z
No. Observations: 1224    Method:             REML            
No. Groups:       161     Scale:              0.4328          
Min. group size:  5       Log-Likelihood:     -1307.0393      
Max. group size:  24      Converged:          No              
Mean group size:  7.6                                         
---------------------------------------------------------------
                 Coef.  Std.Err.    z     P>|z|  [0.025  0.975]
---------------------------------------------------------------
Intercept        0.000     0.045   0.002  0.999  -0.088   0.088
grupa            0.004     0.062   0.062  0.951  -0.118   0.126
theta_pretest_z  0.701     0.020  34.807  0.000   0.662   0.741
Group Var        0.094     0.037                               



In [132]:
VarianceBetweenSchools = result.cov_re.iloc[0, 0]
VarianceWithinSchools = result.scale

ICC = VarianceBetweenSchools / (VarianceBetweenSchools + VarianceWithinSchools)
print(f"Wariancja między szkołami: {VarianceBetweenSchools:.4f}")
print(f"Wariancja wewnątrz szkół (resztowa): {VarianceWithinSchools:.4f}")
print(f"ICC: {ICC:.4f} (czyli {ICC*100:.2f}%)")

Wariancja między szkołami: 0.0940
Wariancja wewnątrz szkół (resztowa): 0.4328
ICC: 0.1784 (czyli 17.84%)
